# AI-Powered Fraud Defense System (Proof-of-Concept)

Capstone project CDL20 by Denise Ye

https://github.com/deniseye-sketch/azureml-example/blob/main/jupyter/anomaly_detection_creditcard.ipynb


### Executive Summary:

To safeguard our customers and minimize financial loss from unauthorized transactions, we have developed a Machine Learning (ML) Proof-of-Concept (PoC). This system utilizes advanced pattern recognition to identify fraudulent activity in real-time. By transitioning from manual, rule-based detection to an automated, Azure-backed AI model, we can intercept suspicious transactions before they impact customer balances. This PoC is delivered via a centralized GitHub repository containing a Jupyter Notebook, which serves as both the functional engine and a transparent technical roadmap. By orchestrating a complete 'data-to-decision' pipeline—covering data preparation, model training, and performance evaluation—this system provides a scalable framework designed to minimize financial loss and maintain consumer trust without disrupting legitimate transactions.

### What this Proof-of-Concept achieves for the business
- **`Automated Intelligence:`** Moves beyond rigid "if-then" rules to a system that "learns" what normal vs. fraudulent behavior looks like for our specific customers.

- **`End-to-End Transparency:`** Every step of the logic, from how we handle sensitive data to how the model makes a "fraud" call, is documented and repeatable.

- **`Azure-Powered Scalability:`** By leveraging cloud infrastructure, the system is designed to handle our growing transaction volumes with minimal lag.

![Pipline](Pipeline.png)


### Azure ML Component for this PoC Project
|Azure ML Component                                 | Role in the Fraud Detection Process                    |
|---------------------------------------------------|--------------------------------------------------------|
|Azure Machine Learning Workspace                   | The "Command Center" where all data, code, models, and team collaborations are centralized and managed.|
|Compute Instances / Clusters                       | The "Engine Room" that provides the raw processing power needed to analyze millions of transactions and train the model.|
|Datastore & Data Assets                            | The "Secure Vault" where historical transaction data is stored and versioned, ensuring we are always training on the most relevant info.|
|Azure ML Pipelines                                 | The "Assembly Line" that automates the workflow - From cleaning the data to testing the model - ensuring the process is repeatable and error-free.|
|Experiments & Runs                                 | The "Lab Notebook" that records every test we run, allowing us to compare different versions of the model to find the most accurate one.|
|Model Registry                                     | The "Inventory Shelf" where the "best" version of the fraud model is stored, stamped with approval, and ready for use.|
|Managed Endpoints                                  | The "Service Counter" where the model lives. It receives a live transaction request and instantly replies with a  "Fraud" or "Safe" score.     | 
|GitHub Integration                                 | The "Blueprints & Audit Trail" that stores the Jupyter Notebook, allowing for version control and transparent peer review of the code.|                                       


## Workflow

### Step 1: import the PyPi packages


In [ ]:
# Step 1: Import Packages and Connect to your Azure Workspace
from azureml.core import Workspace, Dataset         # see https://pypi.org/project/azureml-core/
import pandas as pd                                 # see https://pandas.pydata.org/docs/
from sklearn.ensemble import IsolationForest        # see https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html
from sklearn.metrics import classification_report   # see https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html
from azureml.core.model import Model                # see https://docs.microsoft.com/en-us/python/api/azureml-core/azureml.core.model?view=azure-ml-py 

### Step 2: Load the Credit Card Fraud Dataset from Azure ML

Retrieve the dataset from our existing workspace, and set this up for use with Pandas.

**IMPORTANT: be mindful of the size of the dataset that you're working with. For example, if you run this notebook locally then be aware that you're downloading around 150Mib from your Azure workspace. When running locally this snippet will take approximately 4 minutes to run.**

#### What This Code Does

- **`Workspace.from_config()`** connects to your Azure ML workspace using the `config.json` file (you should already have this if you followed earlier lectures).
- **`Dataset.get_by_name(...)`** loads the dataset you previously uploaded and registered in the Azure ML web interface.
- **`.to_pandas_dataframe()`** converts the Azure Dataset into a standard pandas DataFrame so you can explore and manipulate it with Python.
- **`df.head()`** shows the first 5 rows of the data — this is just a quick preview to confirm that the dataset loaded correctly.

#### Why This Matters

This is the standard pattern you’ll use throughout Azure ML when working with registered datasets in notebooks. It keeps your workflow consistent and lets you:
- Avoid re-uploading data every time.
- Ensure reproducibility across experiments and pipelines.
- Easily switch to remote compute environments without changing your code.

#### Console output

You might (probably) see a few console output messages. This is expected. They come from Azure’s background systems for logging and monitoring.  
Unless you see an actual `ERROR` or `Traceback`, you can **safely ignore** any of the following.

- **`Warning: Falling back to use azure cli login credentials.`**  
  - Azure is using your Azure CLI login (`az login`) for authentication.
  - ✅ This is normal and expected for local development.
  - ⚠️ For production, consider using `ServicePrincipalAuthentication` or `MsiAuthentication`.

- **`{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}`**  
  - The dataset is being converted to a pandas DataFrame.
  - Azure is not auto-detecting column types.
  - ✅ This means `to_pandas_dataframe()` is working.

- **`Timeout was exceeded in force_flush().`**  
  - A background telemetry system couldn’t send logging data in time.
  - ✅ This is safe to ignore. It has no effect on your code or data.

- **`Overriding of current TracerProvider / LoggerProvider / MeterProvider is not allowed`**  
  - Azure's telemetry was already initialized; it's skipping a duplicate setup.
  - ✅ This is common and harmless in notebook environments.

- **`Attempting to instrument while already instrumented`**  
  - Azure ML SDK tried to attach diagnostics tools (e.g., to pandas or HTTP), but they were already connected.
  - ✅ This is internal setup noise — not an error.

In Azure AI Machine Learning Studio - Notebooks you might encounter this error:

```console
UserErrorException: UserErrorException:
	Message: The workspace configuration file config.json, could not be found in /synfs/notebook/0/aml_notebook_mount or its parent directories. Please check whether the workspace configuration file exists, or provide the full path to the configuration file as an argument. You can download a configuration file for your workspace, via http://ml.azure.com and clicking on the name of your workspace in the right top.
	InnerException None
	ErrorResponse 
{
    "error": {
        "code": "UserError",
        "message": "The workspace configuration file config.json, could not be found in /synfs/notebook/0/aml_notebook_mount or its parent directories. Please check whether the workspace configuration file exists, or provide the full path to the configuration file as an argument. You can download a configuration file for your workspace, via http://ml.azure.com and clicking on the name of your workspace in the right top."
    }
}```



In [ ]:
# You only need to run this if you've imported this notebook to Azure AI Machine Learning Studio - Notebook,
# in which case you'll also need to upload the config.json file to the same directory as this notebook,
# and then execute this code to determine the current working directory.
import os
print("Current working directory:", os.getcwd())
print("Files in this directory:", os.listdir())


In [ ]:
# if you're running locally then use this ...
path = None

# alternatively, if you're running in Azure AI Machine Learning Studio - Notebook, then use this ...
# (make sure to upload the config.json file to the same directory as this notebook)
#  and then execute this code to determine the current working directory.
path='Users/[REPLACE-THIS-WITH-YOUR-USERNAME]/config.json'
ws = Workspace.from_config(path=path)
dataset = Dataset.get_by_name(ws, name='creditcard_fraud')
df = dataset.to_pandas_dataframe()
df.head()

### Step 3: Prepare the Data

we're going to normalize the distribution of the transaction $ amount column, which helps the model treat transaction amounts on the same scale as the other features (which are already normalized).

#### What This Code Does

- **Standardizes the `Amount` column**:  
  We scale the `Amount` feature so that it has a mean of 0 and a standard deviation of 1.  
  
- **Creates feature and label sets**:
  - `X` contains the features used to make predictions.
  - `y` contains the target variable: `Class` (where `1 = fraud` and `0 = normal`).

- We also drop the `Time` column since it doesn't contribute meaningfully to anomaly detection in this context.

#### Why This Matters

Many machine learning algorithms — including Isolation Forest — perform better when numeric features are on a similar scale.  
Also, splitting the data into `X` and `y` is a standard step that prepares it for training and evaluation.

In [ ]:
df['Amount'] = (df['Amount'] - df['Amount'].mean()) / df['Amount'].std()
X = df.drop(columns=['Class', 'Time'])
y = df['Class']

### Step 4: Train the model

The **Isolation Forest** algorithm is a popular unsupervised method for **detecting anomalies** in high-dimensional datasets. Instead of learning what “normal” looks like, it works by **isolating outliers** — rare points that are easier to separate from the rest of the data. It does this by randomly splitting the dataset using decision trees and measuring how quickly a data point can be isolated. The idea is that **anomalies require fewer splits to isolate**, because they are different from everything else. Isolation Forest is widely used in **fraud detection**, **network security**, and **industrial monitoring** because it is **fast, efficient**, and handles **large datasets** with many features. In our code, we set the `contamination` parameter to roughly match the known fraction of fraud cases in the dataset.

In [ ]:
model = IsolationForest(contamination=0.0017, random_state=42)
model.fit(X)
y_pred = model.predict(X)
y_pred = [1 if x == -1 else 0 for x in y_pred]

### Step 5: Evaluating the Anomaly Detection Model

The table below is a summary of metrics that are calculated from the [confusion matrix](https://en.wikipedia.org/wiki/Confusion_matrix). It shows how well our model identified normal and fraudulent transactions:

| Metric       | What It Means                                                                 |
|--------------|--------------------------------------------------------------------------------|
| **Precision** | How often the model was *correct* when it said a transaction was fraud        |
| **Recall**    | How many of the *actual fraud cases* the model successfully found             |
| **F1-Score**  | A balance between precision and recall — like a combined performance score     |
| **Support**   | The number of examples in each group (normal or fraud) in the real data        |

#### Results Summary

| Class | Description           | Precision | Recall | F1-Score | Support |
|-------|------------------------|-----------|--------|----------|---------|
| `0`   | Normal transactions    | **1.00**  | **1.00** | **1.00**   | 284,315 |
| `1`   | Fraudulent transactions| **0.29**  | **0.28** | **0.28**   | 492     |

#### Interpretation (In Simple Terms)

- The model is **excellent at recognizing normal transactions** — it almost never makes a mistake with those.
- However, it **struggles to correctly catch fraud**:
  - When it says a transaction is fraud, it’s **only right 29% of the time**.
  - It **only finds 28% of the real fraud cases** — it misses most of them.

#### Overall Accuracy

- The model is **99.9% accurate**, but this is misleading.
- Because **fraud cases are very rare**, the model can look “perfect” just by saying everything is normal.
- That’s why we look at **precision**, **recall**, and **F1-score** for a fuller picture.


In [ ]:
# Step 5: Evaluate Model
print(classification_report(y, y_pred))

### Step 6 (Optional): Register the Model


In [ ]:
import joblib                                       # see https://joblib.readthedocs.io/en/latest/
                                                    #     Joblib is a set of tools to provide lightweight pipelining in Python
joblib.dump(model, 'isolation_forest.pkl')
Model.register(model_path='isolation_forest.pkl',
               model_name='creditcard_if_model',
               workspace=ws)


### Step 7: Visualize a Count of Predicted Anomalies

The chart below is a typical summarization of an anomaly detection analysis. It shows how many transactions the model predicted as **normal (0)** and **anomalies/fraud (1)**:

- **X-axis**: The prediction labels.
  - `0` means the model thinks the transaction is **normal**.
  - `1` means the model thinks the transaction is **fraud** or **anomalous**.
- **Y-axis**: The total number of transactions in each category.

#### How to Interpret This Chart

- You will (hopefully) see a **very tall bar for `0`** and a **very short bar for `1`**.
- This is because **fraud is rare** in the dataset (only 492 out of 284,807 transactions).
- The model is trained to detect outliers, so it **flags a small number of transactions as anomalies** (which is expected).
- If the number of predicted frauds is **close to the actual number** (around 500), that’s a good sign that the model is well-calibrated.

#### Why This Matters

- This simple chart gives a **quick health check** of how aggressive or conservative the model is in flagging anomalies.
- If the model predicts **too many anomalies**, it might be overreacting.
- If it predicts **almost none**, it might be too cautious — missing fraud cases.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Add predictions to the original dataframe
df['predicted_anomaly'] = y_pred

# Count of predicted anomalies
sns.countplot(x='predicted_anomaly', data=df)
plt.title('Count of Predicted Anomalies')
plt.xlabel('Anomaly (1) vs Normal (0)')
plt.ylabel('Count')
plt.show()


### Step 7 (continued): Visualize Transaction Amount by Prediction Class

The boxplot below compares the **amount of money** in transactions that the model predicted as **normal (0)** or **anomalous/fraud (1)**.

- **X-axis**: The model’s prediction.
  - `0` = predicted normal transaction
  - `1` = predicted fraud/anomaly
- **Y-axis**: The dollar **amount** of each transaction (standardized)

#### How to Interpret This Chart

- Each box shows how transaction amounts are distributed for each prediction class.
- The **line in the middle** of each box is the **median** transaction amount.
- The **height of the box** shows where most transaction amounts fall.
- **Dots outside the box** are **outliers** — unusual values far from the average.

#### What This Tells Us

- You may see that predicted frauds (`1`) tend to have **more extreme** or **variable amounts**.
- This could suggest that the model is flagging **unusually high or low transaction amounts** as suspicious.
- If the fraud predictions have a **much wider range**, it means the model may be reacting to extreme values — which is common in anomaly detection.

#### Usefulness

This chart helps you:
- Understand what kinds of amounts the model thinks are suspicious.
- Spot any bias in the model (e.g. only flagging large transactions).
- Decide whether you need to normalize, transform, or engineer features differently.

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='predicted_anomaly', y='Amount')
plt.title('Transaction Amount by Prediction Class')
plt.show()


### Step 7 (continued): SHAP Beeswarm Plot – Feature Importance for Anomaly Detection

The beeswarm plot below is generated using **SHAP** (SHapley Additive exPlanations). It helps explain **which features influenced the model's decisions**, and **how strongly**. We only analyze the first 100 transactions here in order to keep the visualization fast and readable.

#### How to Read the SHAP Beeswarm Plot

- **Each dot** represents a single transaction.
- **Each row** is one feature (like `V1`, `V2`, `Amount`, etc.).
- **Color** shows the feature value for that transaction:
  - **Red = high** value
  - **Blue = low** value
- **Horizontal position** shows **impact on the model’s prediction**:
  - Dots farther to the right **push the model toward predicting fraud**.
  - Dots farther to the left **push the model toward predicting normal**.

#### What This Tells Us

- The **topmost features** are the most important ones in the model’s decisions.
- For example, if `V14` is at the top and its red dots are far right, it means:
  - High values of `V14` increase the chance that the model flags a transaction as fraud.
- This plot helps us understand **why** the model flagged certain transactions as anomalies.

#### Why Use SHAP?

- SHAP adds transparency to the model, even for complex algorithms like Isolation Forest.
- Helps **build trust**, especially in sensitive tasks like fraud detection.
- Guides feature selection and **future model improvements**.

In [ ]:
import shap

explainer = shap.Explainer(model, X)
shap_values = explainer(X[:100])
shap.plots.beeswarm(shap_values)

## Business Impact Assessment
### Cost-benefit analysis
In machine learning, no model is 100% perfect. We must tune our Proof-of-Concept to balance two specific types of errors that have vastly different impacts on our bottom line.

|             | Cost        |  Loss         | Risk       | Impact        |
|-------------|-------------|---------------|------------|---------------|
| False Positive  (The "Insult" Factor) | Increased call center volume and manual investigation time | he immediate loss of the transaction commission/fee | Customer Churn. High-value customers who experience "transaction insult" are likely to switch to a competitor | High friction, brand damage, and administrative overhead |
| False Negative  (The "Leakage" Factor) | The actual dollar amount stolen from the account | Potential fines for inadequate anti-money laundering (AML) controls or security failures and the expense of the chargeback process and reimbursing the victim | May lost customer / business trust | Direct hit to capital, decreased investor confidence, and regulatory scrutiny |


**Economic Trade-off Matrix**
|Error Type|Technical Term|Business Consequence|Primary Goal|
|----------|----------------|-------------------|--------------------------------------------|
|False Positive | Type 1 Error |Customer Dissatisfaction | Maintain "Low Friction" |
|False Negative |Type 2 Error | Direct Financial Loss| Maintain "High Security"|


### Recommendations for model improvement and deployment

Strategic Recommendation for StakeholdersFor this Proof-of-Concept, we utilize a Cost Function during model training. This means we don't just tell the model to "be accurate"; we tell it that missing "$10,000" in fraud is 50x more expensive than accidentally flagging a "$100" legitimate purchase.By adjusting the "Classification Threshold" in our Azure ML Pipeline, we can shift the model to be more conservative (saving more money) or more permissive (keeping customers happier), depending on the current quarterly business priority.

1. **For Model Improvement**

We recommend the following technical enhancements to increase the "Alpha" (competitive edge) of our detection:
  - **Graph-Based Feature Engineering:** Integrate Azure Cosmos DB (Graph API) to identify "Fraud Rings." By analyzing the relationships between disparate accounts (e.g., shared IP addresses, phone numbers, or devices), we can detect coordinated attacks that look like normal individual transactions.
  - **Real-Time Behavioral Profiling:** Shift from "Point-in-Time" analysis to "Velocity Tracking." This involves calculating features on the fly, such as: "Has this user attempted more than 5 transactions in the last 10 minutes?"
  - **Ensemble Stacking:** Move beyond a single model. We recommend a Challenger-Champion setup where a "Deep Learning" model (high sensitivity) and a "Random Forest" model (high stability) vote on the final score to reduce False Positives.
  - **Explainable AI (XAI) Integration:** Use Azure Machine Learning Interpretability (SHAP values) to provide a "reason code" for every flagged transaction. This allows our manual review team to understand why the AI made a decision, drastically reducing investigation time.

2. **For Deployment Strategy (The MLOps Roadmap)**

    For a financial services environment, the deployment must be Resilient, Scalable, and Auditable. We propose a three-phased rollout using Azure Machine Learning Managed Endpoints:

    | Phase | Action | Goal |
    |------------|----------|--------|
    | Phase 1 Shadow Deployment (Observation Mode) | Deploy the model to a production-parallel environment where it receives live data but does not block transactions | Validate the model's "Live vs. Training" performance and calculate the real-world impact of False Positives without risking customer experience |
    | Phase 2 Canary Rollout (Controlled Launch) | Route 5% of transaction traffic to the new model while the remaining 95% stays on the legacy system | Test the Azure Kubernetes Service (AKS) infrastructure for latency (ensuring sub-50ms response times) and system stability under load |
    | Phase 3 Full Production with Continuous Training (CT) | Automate the Azure ML Pipeline to retrain the model weekly using the latest labeled data from our fraud investigators | Ensure the model never "decays." If the retraining shows a performance drop, the system will automatically roll back to the previous "Champion" version |

   
    **Summary Table: Deployment Readiness**
    

    |  Feature | Current PoC | Production Recomendation |
    |----------|-------------|--------------------------|
    | Compute | Jupyter Notebook	| Azure Kubernetes Service (AKS) |
    | Data Flow	| Batch CSV	| Azure Event Hubs (Real-time Stream) |


### Risk assessment and mitigation strategies

|Risk | Thread  | Business Impact |  Mitigation Strategy | Other concern |
|-----|---------|-----------------|----------------------|---------------|
| Data Security & Privacy Risk | Unauthorized access to sensitive customer PII (Personally Identifiable Information) during the training or inference phase | Regulatory fines (GDPR/CCPA), loss of customer trust, and legal liability | Azure Key Vault: Store all connection strings and secrets securely rather than in the code | 1) Data Masking: Use anonymized datasets where names and account numbers are replaced with synthetic identifiers        2) Role-Based Access Control (RBAC): Restrict access to the Azure ML Workspace to only authorized data scientists |
| Model Drift & Degradation | Fraudsters constantly change their tactics. A model trained on 2024 data may become obsolete by late 2025 | A steady increase in missed fraud (False Negatives) over time, leading to unexpected financial losses | Continuous Monitoring: Implement Azure ML monitoring to alert the team when "Data Drift" occurs (when incoming live data looks significantly different from training data) | Automated Retraining: Use the Azure ML Pipeline to automatically trigger a new training run when performance dips below a specific threshold |
| Algorithmic Bias & Fairness | The model may inadvertently learn to flag transactions based on protected characteristics (e.g., geography or age) rather than actual criminal behavior | Reputational damage and potential "Redlining" lawsuits or regulatory audits | Fairlearn Integration: Use the Azure Fairlearn toolkit to check the model for disparate impacts across different demographic groups before deployment | Feature Importance Transparency: Use Azure Machine Learning Interpretability (SHAP/LIME) to explain why a transaction was flagged, ensuring the logic is based on financial patterns, not identity |
| System Latency & Availability | If the model takes too long to respond, it slows down the checkout process for the customer | High cart abandonment rates and a poor user experience | Managed Online Endpoints: Deploy the model using high-performance Azure compute (like AKS) to ensure sub-second response times | Fallback Logic: Implement a "Fail-Open" or "Rule-Based" fallback so that if the AI service is momentarily down, transactions are still processed via traditional safety rules |


### Stakeholder communication plan
Here is a 90 days communication Roadmap

| Phase                             | Objective     | Key Event         |
|-----------------------------------|---------------|-------------------|
| Phase 1 The "Proof" (Weeks 1–4) | Socialize the PoC results |  "The Transparent AI Demo." Show the GitHub repository and walk stakeholders through a "Success Story"—a complex fraudulent transaction the model caught that the old rules missed. |
| Phase 2: The "Pilot" (Weeks 5–8) | Build trust in the "Shadow Mode" results | Monthly ROI Report. Present the "Theoretical Savings" vs. "Actual Manual Effort," highlighting how much money would have been saved if the model was live. |
| Phase 3: The "Production Path" (Weeks 9–12) | Secure budget for full Azure deployment | Deployment Readiness Review. A final sign-off meeting with IT and Risk to confirm the mitigation strategies (discussed previously) are in place. |

### Escalation & Feedback Loop
   - **Must have a Human-in-the-loop** for feedback
     - Channel: dedicated Team/slock/Zoom channel
     - Process - Flag for "False Positive and "re-training" at the next update, it prove the system is accountable, not working in black box.